# Chapter 8 — Comparisons and Logical Operations

Part 3: Small mathematical operations.

**Ladder:** Python predicates and masks (Method 01) → Triton comparison + `tl.where` kernels (Method 02) → fused mask-select, dead-zone, and soft-threshold kernels (Method 03).

**Study tip.** Take

$$
x = (-2,\,-1,\,-0.5,\,0,\,0.5,\,1,\,2)
$$

and write the six masks $[x=0]$, $[x\ne0]$, $[x\gt 0]$, $[x\ge1]$, $[x\lt 0]$, $[x\le-1]$ by hand before you read any code. Then write the band mask $[-1\lt x\lt 1]$ and its complement. If your hand-written rows match the next numerical cell, the rest of the chapter is implementation detail.

**Learning objectives**

- Read a comparison as an elementwise map from a tile of numbers to a tile of predicates.
- State the **same-shape rule** and the **broadcast rule** for comparisons, masks, and `tl.where`.
- Build a band mask, its complement, and a multi-condition mask with parentheses and De Morgan's laws.
- Select values with `tl.where` instead of a per-lane branch, and explain why both arms are evaluated.
- Write a Triton kernel that stores a mask, selects values, and fuses a mask with arithmetic.
- Derive the backward pass of `where` and of a piecewise-linear mask.

[Index](../../00_index.ipynb) · [Chapter README](README.md) · [← Previous](../ch07_arithmetic/07_complete.ipynb) · [Next →](../ch09_activations/09_complete.ipynb)

**Estimated time:** 55–70 min · **Difficulty:** Intermediate

**Prerequisites:** Chapter 7 (elementwise arithmetic, masks, `other=`); basic Boolean algebra.

**Book apparatus:** [Setup](../../docs/SETUP.md) · [Notation](../../docs/NOTATION.md) · [Glossary](../../docs/GLOSSARY.md) · [API index](../../docs/API_INDEX.md) · [Formula index](../../docs/FORMULA_INDEX.md) · [Diagrams](../../docs/DIAGRAMS.md) · [Troubleshooting](../../docs/TROUBLESHOOTING.md) · [Exercise hints](../../docs/SOLUTIONS.md)

## Chapter map

| § | Topic | Key formula | Shape rule | Applied kernel |
|---|---|---|---|---|
| **8.1** | Equal | $m_i=[x_i=y_i]$ | same shape, or broadcast | six-comparison bit code |
| **8.2** | Not equal | $m_i=[x_i\ne y_i]$ | same shape, or broadcast | six-comparison bit code |
| **8.3** | Greater than | $m_i=[x_i\gt y_i]$ | same shape, or broadcast | six-comparison bit code |
| **8.4** | Greater or equal | $m_i=[x_i\ge y_i]$ | same shape, or broadcast | six-comparison bit code |
| **8.5** | Less than | $m_i=[x_i\lt y_i]$ | same shape, or broadcast | six-comparison bit code |
| **8.6** | Less or equal | $m_i=[x_i\le y_i]$ | same shape, or broadcast | six-comparison bit code |
| **8.7** | Logical AND | $m_i=a_i\land b_i$ | same shape, or broadcast | Boolean mask combinations |
| **8.8** | Logical OR | $m_i=a_i\lor b_i$ | same shape, or broadcast | Boolean mask combinations |
| **8.9** | Logical NOT | $m_i=\lnot a_i$ | unchanged | Boolean mask combinations |
| **8.10** | Boolean masks | $m_i\in\{0,1\}$ | mask shape = guarded tile shape | masked load/store |
| **8.11** | `tl.where()` | $y_i=c_i\,a_i+(1-c_i)\,b_i$ | broadcast of `cond`, `a`, `b` | conditional selection |
| **8.12** | Combining conditions | $m_i=a_i\land b_i\land c_i$ | same shape, or broadcast | dead-zone filter |
| **8.13** | Runnable mask/select | $y_i=m_i a_i+(1-m_i)b_i$ | same shape | soft-threshold |
| **★** | Capstone | $y_i=x_i/\hat d_i$ | same shape | masked normalization |

**One-line thesis.** A comparison does not branch; it produces a tile of predicates. A mask does not change control flow; it guards lanes. `tl.where` does not branch; it selects lane by lane.

In [ ]:
import sys
from pathlib import Path

root = Path.cwd()
if (root / "00_Basic").exists():
    root = root / "00_Basic"
while root.name != "00_Basic" and root.parent != root:
    root = root.parent
sys.path.insert(0, str(root))

import pytest
import torch
import triton
import triton.language as tl
from triton.language.extra import libdevice

from ttools import DEVICE, HAS_GPU, check, require_gpu, run_if_gpu

print("device:", DEVICE, "| GPU:", HAS_GPU, "| triton:", triton.__version__)

## Concept diagrams

### Comparison → mask → select

```text
      x tile                         y tile
        |                              |
        +----------> compare <---------+
                       |
                       v
                 predicate tile  m in {0,1}
                    /          \
                   |            |
                   v            v
             tl.load/store    tl.where(m, a, b)
             (guard lanes)    (choose lanes)
```

### The tile is the unit

```text
program pid = 1, BLOCK = 4, N = 7

 flat index:   0 1 2 3 | 4 5 6 7
               ---------+---------
 offs:         0 1 2 3 | 4 5 6 7
 mask:         1 1 1 1 | 1 1 1 0
 lane exists:  y y y y | y y y n
```

The last lane is **not** a second-class value; it is a lane with `mask=0`. The
comparison may still execute there, but the load and the store must not touch it.

# Part I — Mathematics

**Thread of the chapter.** Every mask in this chapter is built from one
comparison. A comparison is an elementwise lift whose codomain is
$\{0,1\}$, not $\mathbb R$. The six predicates differ only at their
boundaries; the logical operators combine predicates; `tl.where` turns a
predicate back into a value. Keep that chain in mind:

- **compare** two tiles $\to$ a predicate tile;
- **combine** predicate tiles with `&`, `|`, `~`;
- **guard** a load or store with the predicate;
- **select** a value with `tl.where`;
- **fuse** the predicate with arithmetic in one kernel.

Each section below adds one link to that chain and says which earlier link it
uses.

## 1. The comparison contract

**Connection.** Chapter 7 treated $f:\mathbb R\to\mathbb R$ lifted to a tile.
A comparison is the same lift with a different codomain.

- **Rule.** $m_i=[x_i\bowtie y_i]$ for $\bowtie\in\{=,\ne,\gt ,\ge,\lt ,\le\}$.
- **Same-shape rule.** If both operands are full tensors, every dimension must
  match; the result has the same shape.
- **Broadcast rule.** If the shapes differ, right-align the axes; each axis must
  have equal sizes or one of them must be $1$. The result has the broadcast shape.
- **No third value.** An ordinary comparison is true or false. `nan` is the
  exception: ordered comparisons with `nan` are false, while `!=` is true.
- **Kernel rule.** The comparison may run on a masked lane; only the load/store
  is suppressed. A predicate is not a branch.
- **Why it matters.** A warp executes one instruction stream. A predicate tile
  keeps the warp converged; a per-lane branch does not.

## 2. The six comparisons

**Connection.** All six are the same lift from §1 with a different relation
$\bowtie$. Read them as one family, not six unrelated operators.

**The spine.** Take $x=(-2,-1,-0.5,0,0.5,1,2)$ and $y=0$. Each predicate below
returns a length-7 mask.

| § | Predicate | Rule | Mask on the spine | Where it appears |
|---|---|---|---|---|
| **8.1** | Equal | $[x_i=y_i]$ | `F F F T F F F` | one-hot, id checks |
| **8.2** | Not equal | $[x_i\ne y_i]$ | `T T T F T T T` | exclusion, masking |
| **8.3** | Greater than | $[x_i\gt y_i]$ | `F F F F T T T` | ReLU gates |
| **8.4** | Greater or equal | $[x_i\ge y_i]$ | `F F F T T T T` | lower bounds, valid ids |
| **8.5** | Less than | $[x_i\lt y_i]$ | `T T T F F F F` | upper bounds |
| **8.6** | Less or equal | $[x_i\le y_i]$ | `T T T T F F F` | inclusive windows |

### 8.1 Equal — the mask that isolates a value

- **Connection.** This is the first concrete predicate; §8.2 is its complement.
- **Rule.** $m_i=[x_i=y_i]$; shape is the broadcast shape of $x$ and $y$.
- **Where.** One-hot construction, exact id lookup, exact-match filters.
- **Triton.** `x == y`; cast to `int8` before storing a mask.
- **Hand check.** On the spine, only $x=0$ is true: `F F F T F F F`.
- **Failure.** Float equality is exact. `0.1 + 0.2 == 0.3` is `False`. For a
  "close" test use `tl.abs(x-y) <= eps`.

### 8.2 Not equal — the complement

- **Connection.** $\lnot$ of §8.1; the two masks partition the spine.
- **Rule.** $m_i=[x_i\ne y_i]$.
- **Where.** Excluding a token id, selecting every class except one.
- **Triton.** `x != y`.
- **Hand check.** `T T T F T T T`.
- **Failure.** `nan != nan` is true, so a `!=` filter keeps NaNs. State the
  NaN policy explicitly.

### 8.3 Greater than — a strict lower gate

- **Connection.** A strict version of the lower-bound mask used by ReLU in
  Chapter 9.
- **Rule.** $m_i=[x_i\gt y_i]$; the boundary $x_i=y_i$ is excluded.
- **Where.** Positive-value filters, thresholds, gates.
- **Triton.** `x > y`.
- **Hand check.** `F F F F T T T`.
- **Failure.** Mixing `>` and `>=` changes one lane; write a boundary test.

### 8.4 Greater or equal — an inclusive lower bound

- **Connection.** §8.3 with the boundary included; together they answer "at
  least" versus "more than".
- **Rule.** $m_i=[x_i\ge y_i]$.
- **Where.** Valid indices ($0\le i$), inclusive lower bounds.
- **Triton.** `x >= y`.
- **Hand check.** `F F F T T T T`.
- **Failure.** Off-by-one at the boundary; pair every `>` with a `>=` test.

### 8.5 Less than — a strict upper gate

- **Connection.** The mirror of §8.3; together they form a strict window.
- **Rule.** $m_i=[x_i\lt y_i]$; the bound is excluded.
- **Where.** Index validity ($i\lt N$), upper thresholds.
- **Triton.** `x < y`.
- **Hand check.** `T T T F F F F`.
- **Failure.** An open window and a closed window differ at both edges.

### 8.6 Less or equal — an inclusive upper bound

- **Connection.** §8.5 with the boundary included; it completes the closed
  window used by the dead-zone kernel in §9.
- **Rule.** $m_i=[x_i\le y_i]$.
- **Where.** Clamping, closed intervals, "at most" tests.
- **Triton.** `x <= y`.
- **Hand check.** `T T T T F F F`.
- **Failure.** For a dead zone, `>`/`<` keeps no boundary; `>=`/`<=` keeps both.

**Bridge to §3.** The masks above assume $x$ and $y$ have the same shape. Real
tensors often do not; the next section decides when a comparison is legal.

In [ ]:
# The chapter spine. The values are stated in the markdown; the cell only verifies.
x = torch.tensor([-2.0, -1.0, -0.5, 0.0, 0.5, 1.0, 2.0])
y = torch.zeros_like(x)
assert x.shape == y.shape == (7,)
assert torch.broadcast_shapes(x.shape, y.shape) == (7,)

expected = {
    "eq": [False, False, False, True, False, False, False],
    "ne": [True, True, True, False, True, True, True],
    "gt": [False, False, False, False, True, True, True],
    "ge": [False, False, False, True, True, True, True],
    "lt": [True, True, True, False, False, False, False],
    "le": [True, True, True, True, False, False, False],
}
for op, want in expected.items():
    got = {"eq": x == y, "ne": x != y, "gt": x > y,
           "ge": x >= y, "lt": x < y, "le": x <= y}[op]
    assert got.tolist() == want, op

band_open = (x > -1.0) & (x < 1.0)
assert band_open.tolist() == [False, False, True, True, True, False, False]
assert torch.equal(~band_open, (x <= -1.0) | (x >= 1.0))

code = (
    (x == 0).int() + (x != 0).int() * 2 + (x > 0).int() * 4
    + (x >= 1).int() * 8 + (x < 0).int() * 16 + (x <= -1).int() * 32
)
assert code.tolist() == [50, 50, 18, 1, 6, 14, 14]
print("spine: masks and bit code match the text")

## 3. Broadcasting a comparison

**Connection.** §8.1–8.6 compared same-shape tiles. This section keeps the
same predicates but changes the address calculation when the shapes differ.

- **Rule.** Right-align the shapes. For each axis, the sizes must be equal or
  one of them must be $1$. The result has the broadcast shape.
- **Example.** Let

$$
A=\begin{bmatrix}1&2&3\\4&5&6\end{bmatrix},\qquad
r=\begin{bmatrix}2&3&4\end{bmatrix},\qquad
c=\begin{bmatrix}3\\5\end{bmatrix}.
$$

  Then $A\ge r$ and $A\le c+2$ both broadcast to shape $(2,3)$:

$$
(A\ge r)\land(A\le c+2)=
\begin{bmatrix}0&0&0\\1&1&1\end{bmatrix}.
$$

- **Why the second row passes.** Its values $4,5,6$ lie between the row's lower
  bounds $2,3,4$ and the column's upper bounds $5,7,7$.
- **What breaks.** $(2,3)$ with $(3,2)$ is illegal: the last axes are $3$ and
  $2$, and neither is $1$. The host raises before a kernel launches.
- **Kernel consequence.** Triton does not infer a broadcast from strides. The
  host expands the operand and calls `.contiguous()`, or the kernel indexes it
  explicitly (`offs % N`, a 2-D grid, or a stride argument).
- **Bridge to §4.** Once the shapes agree, the resulting predicate tiles can be
  combined with Boolean operators.

In [ ]:
A = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])
r = torch.tensor([2.0, 3.0, 4.0])
c = torch.tensor([[3.0], [5.0]])

assert A.shape == (2, 3) and r.shape == (3,) and c.shape == (2, 1)
assert torch.broadcast_shapes(A.shape, r.shape) == (2, 3)
assert torch.broadcast_shapes(A.shape, c.shape) == (2, 3)

lower = A >= r
upper = A <= (c + 2.0)
inside = lower & upper
assert lower.shape == upper.shape == inside.shape == (2, 3)
assert inside.int().tolist() == [[0, 0, 0], [1, 1, 1]]

try:
    torch.broadcast_shapes((2, 3), (3, 2))
except RuntimeError:
    pass
else:
    raise AssertionError("(2,3) and (3,2) should not broadcast")

picked = torch.where(A > 3.0, A, torch.tensor([[-1.0], [-2.0]]))
assert picked.shape == (2, 3)
assert picked.tolist() == [[-1.0, -1.0, -1.0], [4.0, 5.0, 6.0]]
print("broadcast example: shapes and masks match the text")

## 4. Boolean algebra for masks

**Connection.** §8.1–8.6 produced individual predicates. This section combines
them; every band, complement, and multi-condition mask in the chapter is built
here.

### 8.7 Logical AND — the intersection of two masks

- **Connection.** Takes two predicates from §8.1–8.6 and keeps only the lanes
  where both are true.
- **Rule.** $m_i=a_i\land b_i$, written `a & b`.
- **Example.** On the spine, $a=[x\gt -1]$ is `F F T T T T T`, $b=[x\lt 1]$ is
  `T T T T T F F`, so $a\land b$ is `F F T T T F F`.
- **Where.** A band mask, a valid-index mask (`0<=i` AND `i<N`), and any
  multi-condition filter.
- **Triton.** `(x > lo) & (x < hi)`; parenthesize both comparisons.
- **Failure.** `and` is scalar short-circuit and raises on tensors; `&` binds
  before the comparisons, so `x > lo & x < hi` is not the band mask.

### 8.8 Logical OR — the union of two masks

- **Connection.** The dual of §8.7; it keeps a lane if either predicate is true.
- **Rule.** $m_i=a_i\lor b_i$, written `a | b`.
- **Example.** For the same $a,b$, $a\lor b$ is all true on the spine, because
  every lane is either $\gt -1$ or $\lt 1$.
- **Where.** The complement of a band, "either condition" filters.
- **Triton.** `(x <= lo) | (x >= hi)`.
- **Failure.** De Morgan is the safer way to write a complement of a compound
  predicate; check it lane by lane.

### 8.9 Logical NOT — the complement of a mask

- **Connection.** The unary operation on the predicates from §8.7–8.8.
- **Rule.** $m_i=\lnot a_i$, written `~a`.
- **Example.** The complement of `F F T T T F F` is `T T F F F T T`.
- **Where.** Excluding a band, inverting a validity mask.
- **Triton.** `~mask`; the operand must be Boolean, not an integer.
- **Failure.** `~` on an integer tensor is a bitwise complement, not a Boolean
  negation. Cast to `bool` first.

### 8.10 Boolean masks — predicates that guard memory

- **Connection.** §8.7–8.9 combine predicates; this section uses the result as
  the `mask=` argument of a load or store.
- **Rule.** A mask is a tile of predicates with the same shape as the guarded
  tile (or broadcastable to it).
- **Rule.** `tl.load(ptr+offs, mask=mask, other=...)` reads only the live lanes;
  `tl.store(..., mask=mask)` writes only the live lanes.
- **`other`.** Choose the neutral element of the operation that follows: `0` for
  a sum, `1` for a product, `-inf` for a max, `+inf` for a min, never `0` for a
  log or a denominator.
- **Where.** Every tail tile; every gather with an invalid index.
- **Failure.** A comparison may execute on a masked lane, but the memory access
  must not. Forgetting the mask on the store writes past the tensor.

**Bridge to §5.** A predicate can guard memory (§8.10) or choose a value
(§8.11); the next section separates those two uses.

In [ ]:
x = torch.tensor([-2.0, -1.0, -0.5, 0.0, 0.5, 1.0, 2.0])
a = x > -1.0
b = x < 1.0
assert a.shape == b.shape == (7,)
assert torch.equal(~(a & b), (~a) | (~b))
assert torch.equal(~(a | b), (~a) & (~b))

band = (x > -1.0) & (x < 1.0)
complement = (x <= -1.0) | (x >= 1.0)
assert band.tolist() == [False, False, True, True, True, False, False]
assert complement.tolist() == [True, True, False, False, False, True, True]
assert torch.equal(band | complement, torch.ones_like(band))
assert not (band & complement).any().item()
print("Boolean algebra: De Morgan and the band partition hold")

## 5. `tl.where`: selection without a branch

**Connection.** §8.10 used a predicate to guard memory. This section uses the
same predicate to choose between two values, which is the value-level analogue
of a branch.

### 8.11 `tl.where()` — conditional selection

- **Rule.** $y_i=c_i a_i+(1-c_i)b_i$, written `tl.where(c, a, b)`.
- **Shape rule.** The output shape is the broadcast of `c`, `a`, and `b`.
- **Example.** With $c=$ `T F T`, $a=(1,2,3)$, and $b=-1$, the result is
  $(1,-1,3)$.
- **Both arms are evaluated.** If an arm divides by zero or overflows, that
  value is computed; guard the input of the dangerous expression, not the output.
- **Backward.** $g=\partial L/\partial y$; then $\partial L/\partial a=g\,c$ and
  $\partial L/\partial b=g\,(1-c)$; the condition has no classical derivative.
- **Failure.** A tensor `if` becomes a mask anyway, and a per-lane branch
  diverges the warp.

### 8.12 Combining conditions — band, complement, piecewise

- **Connection.** This is the composition of §8.7–8.11: a compound predicate
  followed by a select.
- **Rule.** `keep = (x >= lo) & (x <= hi)`; `y = tl.where(keep, x, 0)`.
- **Example.** The closed band on the spine is `F T T T T T F`; the dead-zone
  output is $(0,-1,-0.5,0,0.5,1,0)$.
- **Open vs closed.** `>`/`<` keeps no boundary; `>=`/`<=` keeps both. State
  which one the kernel implements.
- **Where.** Dead-zone filters, gated residuals, safe division.
- **Failure.** Mixing `&` and `|` without parentheses changes the mask; write
  the truth table on the boundary values.

**Bridge to Part II.** The next part writes the scalar reference for all of
these predicates, masks, and selects, then Part III implements them in Triton.

In [ ]:
c2 = torch.tensor([[True, False, True], [False, True, False]])
a2 = torch.tensor([[1.0], [2.0]])
b2 = torch.tensor(-1.0)
y2 = torch.where(c2, a2, b2)

assert c2.shape == (2, 3) and a2.shape == (2, 1) and b2.shape == ()
assert y2.shape == torch.broadcast_shapes(c2.shape, a2.shape, b2.shape) == (2, 3)
assert y2.tolist() == [[1.0, -1.0, 1.0], [-1.0, 2.0, -1.0]]

# Both arms are evaluated: the unselected division still creates a non-finite value.
num = torch.tensor([1.0, 2.0, 3.0])
den = torch.tensor([1.0, 0.0, 2.0])
assert not torch.isfinite(num / den)[1].item()
print("tl.where: broadcast shape and eager arms match the text")

# Part II — Method 01: naive predicates and masks

The reference implementation is deliberately scalar: one Python iteration per
element, one explicit `if` per lane. It is slow on a GPU and it is the
specification the Triton kernels are checked against.

Three functions are enough for the whole chapter:

1. `naive_compare` — six predicates, one element at a time.
2. `naive_where` — broadcast the three operands, then select element by element.
3. `naive_dead_zone` — a band mask and a select.

The naive functions also make the shape rule explicit: before the loop, the
inputs are broadcast with `torch.broadcast_tensors`, and the output is allocated
with the broadcast shape.

In [ ]:
COMPARE_OPS = ("eq", "ne", "gt", "ge", "lt", "le")


def naive_compare(x: torch.Tensor, y: torch.Tensor, op: str) -> torch.Tensor:
    '''Method 01: one scalar comparison per element (the definition).'''
    x, y = torch.broadcast_tensors(x, y)
    out = torch.empty(x.shape, dtype=torch.bool, device=x.device)
    xf, yf, of = x.reshape(-1), y.reshape(-1), out.reshape(-1)
    for i in range(xf.numel()):
        a, b = float(xf[i]), float(yf[i])
        if op == "eq":
            of[i] = a == b
        elif op == "ne":
            of[i] = a != b
        elif op == "gt":
            of[i] = a > b
        elif op == "ge":
            of[i] = a >= b
        elif op == "lt":
            of[i] = a < b
        elif op == "le":
            of[i] = a <= b
        else:
            raise ValueError(op)
    return out


def naive_where(cond: torch.Tensor, a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    '''Method 01: broadcast, then choose one scalar per output lane.'''
    cond, a, b = torch.broadcast_tensors(cond, a, b)
    out = torch.empty(cond.shape, dtype=a.dtype, device=a.device)
    cf, af, bf, of = (
        cond.reshape(-1), a.reshape(-1), b.reshape(-1), out.reshape(-1)
    )
    for i in range(cf.numel()):
        of[i] = af[i] if bool(cf[i]) else bf[i]
    return out


def naive_dead_zone(x: torch.Tensor, lo: float, hi: float) -> torch.Tensor:
    '''Method 01: keep the values inside [lo, hi], zero the rest.'''
    out = torch.empty_like(x)
    xf, of = x.reshape(-1), out.reshape(-1)
    for i in range(xf.numel()):
        of[i] = xf[i] if (float(lo) <= float(xf[i]) <= float(hi)) else 0.0
    return out


x = torch.tensor([-2.0, -1.0, -0.5, 0.0, 0.5, 1.0, 2.0])
print("naive eq :", naive_compare(x, torch.zeros_like(x), "eq").tolist())
print("naive ge :", naive_compare(x, torch.ones_like(x), "ge").tolist())
print("dead zone:", naive_dead_zone(x, -0.5, 0.5).tolist())
print("shape stays:", tuple(naive_dead_zone(x, -0.5, 0.5).shape))

In [ ]:
def test_worked_example() -> None:
    '''Check the exact numbers printed in Part I.'''
    x = torch.tensor([-2.0, -1.0, -0.5, 0.0, 0.5, 1.0, 2.0])
    z = torch.zeros_like(x)

    expected = {
        "eq": [False, False, False, True, False, False, False],
        "ne": [True, True, True, False, True, True, True],
        "gt": [False, False, False, False, True, True, True],
        "ge": [False, False, False, True, True, True, True],
        "lt": [True, True, True, False, False, False, False],
        "le": [True, True, True, True, False, False, False],
    }
    for op, want in expected.items():
        got = naive_compare(x, z, op)
        assert got.tolist() == want, (op, got.tolist(), want)

    code = (
        (x == 0).int() + (x != 0).int() * 2 + (x > 0).int() * 4
        + (x >= 1).int() * 8 + (x < 0).int() * 16 + (x <= -1).int() * 32
    )
    assert code.tolist() == [50, 50, 18, 1, 6, 14, 14]

    band = naive_compare(x, torch.full_like(x, -1.0), "gt") & naive_compare(
        x, torch.ones_like(x), "lt"
    )
    assert band.tolist() == [False, False, True, True, True, False, False]
    print("worked example: six masks, bit code, and band mask match the notes")


def test_naive_reference() -> None:
    '''The naive functions agree with PyTorch on a random 2-D case.'''
    torch.manual_seed(0)
    x = torch.randn(4, 5) * 2.0
    y = torch.randn(4, 5)
    for op, ref in [
        ("eq", x == y), ("ne", x != y), ("gt", x > y),
        ("ge", x >= y), ("lt", x < y), ("le", x <= y),
    ]:
        assert torch.equal(naive_compare(x, y, op), ref), op

    cond = x > 0
    a = torch.randn(4, 1)
    b = torch.tensor(3.0)
    assert torch.allclose(naive_where(cond, a, b), torch.where(cond, a, b))
    assert torch.allclose(naive_dead_zone(x, -0.5, 0.5),
                          torch.where((x >= -0.5) & (x <= 0.5), x, torch.zeros_like(x)))
    print("method 01: all six predicates, where, and dead zone match PyTorch")


test_worked_example()
test_naive_reference()

# Part III — Method 02: Triton comparison and selection kernels

## 6. Grid, tile, and mask

All kernels in this chapter use the 1-D skeleton

$$
\text{offs}=\text{pid}\cdot B+\{0,\ldots,B-1\},
\qquad
\text{mask}=\text{offs}\lt N,
\qquad
P=\left\lceil N/B\right\rceil.
$$

The comparison runs on **every** lane of the tile, including the masked tail.
The load and the store are guarded by the same mask, so the tail never touches
memory. There is no scalar branch and no lane divergence.

**Tile map for $N=7$, $B=4$:** $P=\lceil7/4\rceil=2$.

| `pid` | `offs` | `mask` | lanes that exist |
|---:|---|---|---|
| 0 | 0,1,2,3 | 1,1,1,1 | 0,1,2,3 |
| 1 | 4,5,6,7 | 1,1,1,0 | 4,5,6 |

The tail program has one dead lane. Its comparison is computed but its store is
masked, so the output has exactly $N$ elements.

## 7. The comparison kernel

The six predicates share one kernel with a compile-time selector:

```python
if OP == CMP_EQ:
    result = x == y
elif OP == CMP_NE:
    result = x != y
...
```

`OP` is a `tl.constexpr`, so the JIT resolves the branch and emits only the
chosen predicate. There is no runtime switch and no cost for the unused arms.

The output is stored as `int8`, one byte per mask, because a mask is usually
consumed by a later kernel or written to a file. Storing a Triton predicate
directly is possible only with an explicit cast; `result.to(tl.int8)` makes the
storage width part of the code.

In [ ]:
CMP_EQ, CMP_NE, CMP_GT, CMP_GE, CMP_LT, CMP_LE = range(6)


@triton.jit
def m8_compare_kernel(x_ptr, y_ptr, out_ptr, n_elements,
                      OP: tl.constexpr, BLOCK: tl.constexpr):
    pid = tl.program_id(0)
    offs = pid * BLOCK + tl.arange(0, BLOCK)
    mask = offs < n_elements

    x = tl.load(x_ptr + offs, mask=mask, other=0.0)
    y = tl.load(y_ptr + offs, mask=mask, other=0.0)

    if OP == CMP_EQ:
        result = x == y
    elif OP == CMP_NE:
        result = x != y
    elif OP == CMP_GT:
        result = x > y
    elif OP == CMP_GE:
        result = x >= y
    elif OP == CMP_LT:
        result = x < y
    else:
        result = x <= y

    # A Triton predicate is an i1; store one byte per lane explicitly.
    tl.store(out_ptr + offs, result.to(tl.int8), mask=mask)


def compare(x, y, op, BLOCK=256):
    '''Method 02: six predicates, one compiled kernel per OP.'''
    x = torch.as_tensor(x)
    y = torch.as_tensor(y)
    x, y = torch.broadcast_tensors(x, y)
    x = x.contiguous()
    y = y.contiguous()
    out = torch.empty(x.numel(), device=x.device, dtype=torch.int8)
    m8_compare_kernel[(triton.cdiv(x.numel(), BLOCK),)](
        x, y, out, x.numel(), OP=op, BLOCK=BLOCK
    )
    return out.reshape(x.shape)


def test_compare_kernel():
    require_gpu("m8 comparison kernel")
    torch.manual_seed(0)
    x = torch.randn(1000, device=DEVICE)
    y = torch.randn(1000, device=DEVICE)
    refs = [x == y, x != y, x > y, x >= y, x < y, x <= y]
    for op, ref in enumerate(refs):
        check(f"comparison op {op}", compare(x, y, op), ref.to(torch.int8))

    # The tail of the last tile is exercised by N=7 with BLOCK=4.
    xs = torch.tensor([-2.0, -1.0, -0.5, 0.0, 0.5, 1.0, 2.0], device=DEVICE)
    check("tail comparison", compare(xs, torch.zeros_like(xs), CMP_EQ, BLOCK=4),
          (xs == 0).to(torch.int8))


run_if_gpu(test_compare_kernel)

## 8. The `tl.where` kernel

The selection kernel reads three tiles — a condition tile and two value tiles —
and writes one. In this example the condition is computed from `x_ptr` and
`y_ptr`, so the kernel has four pointers:

$$
c=\operatorname{compare}(x,y),
\qquad
z=\operatorname{where}(c,a,b).
$$

The kernel requires the four tiles to have the **same length** (or to be
explicitly broadcast by the host). It does not infer a broadcast from strides;
Triton buffers are raw pointers, and the kernel only knows the offsets it is
given. For a bias of shape `(N,)` added to a matrix `(M,N)`, index the bias with
`offs % N` or launch a 2-D grid; do not expect the kernel to broadcast a
separate pointer automatically.

**`other` and the mask.** The load uses `other=0.0`; the masked lanes are never
stored, so the value is irrelevant for correctness. It would matter if the
selected value were used in a reduction. The rule from Chapter 7 still applies:
`other` is the neutral element of the operation that follows, not a habit.

In [ ]:
@triton.jit
def m8_where_kernel(x_ptr, y_ptr, a_ptr, b_ptr, out_ptr, n_elements,
                    OP: tl.constexpr, BLOCK: tl.constexpr):
    pid = tl.program_id(0)
    offs = pid * BLOCK + tl.arange(0, BLOCK)
    mask = offs < n_elements

    x = tl.load(x_ptr + offs, mask=mask, other=0.0)
    y = tl.load(y_ptr + offs, mask=mask, other=0.0)
    a = tl.load(a_ptr + offs, mask=mask, other=0.0)
    b = tl.load(b_ptr + offs, mask=mask, other=0.0)

    if OP == CMP_GT:
        cond = x > y
    elif OP == CMP_GE:
        cond = x >= y
    elif OP == CMP_LT:
        cond = x < y
    else:
        cond = x <= y

    z = tl.where(cond, a, b)
    tl.store(out_ptr + offs, z, mask=mask)


def where_kernel(x, y, a, b, op=CMP_GT, BLOCK=256):
    '''Method 02: select a where the predicate x op y is true, else b.'''
    x, y, a, b = torch.broadcast_tensors(
        torch.as_tensor(x), torch.as_tensor(y),
        torch.as_tensor(a), torch.as_tensor(b),
    )
    x, y, a, b = (t.contiguous() for t in (x, y, a, b))
    out = torch.empty_like(a)
    m8_where_kernel[(triton.cdiv(a.numel(), BLOCK),)](
        x, y, a, b, out, a.numel(), OP=op, BLOCK=BLOCK
    )
    return out.reshape(a.shape)


def test_where_kernel():
    require_gpu("m8 where kernel")
    torch.manual_seed(0)
    x = torch.randn(1000, device=DEVICE)
    y = torch.zeros_like(x)
    a = torch.randn(1000, device=DEVICE)
    b = torch.randn(1000, device=DEVICE)
    check("where x>0", where_kernel(x, y, a, b, CMP_GT),
          torch.where(x > 0, a, b))
    check("where x<=0", where_kernel(x, y, a, b, CMP_LE),
          torch.where(x <= 0, a, b))

    # A 2-D broadcast: a is (3,1), b is scalar.
    x2 = torch.randn(3, 4, device=DEVICE)
    a2 = torch.randn(3, 1, device=DEVICE)
    check("where broadcast", where_kernel(x2, torch.zeros_like(x2), a2, 0.0, CMP_GT),
          torch.where(x2 > 0, a2, torch.zeros_like(x2)))


run_if_gpu(test_where_kernel)

## 9. A mask kernel: dead zone

The dead-zone filter is the first composition:

$$
y_i =
\begin{cases}
x_i, & l\le x_i\le u,\\
0, & \text{otherwise}.
\end{cases}
$$

It is a band mask followed by a select. Written with comparisons, it is

```python
keep = (x >= lo) & (x <= hi)
y = tl.where(keep, x, 0.0)
```

The mask is a tile, the select is a tile, and the output has the input shape.
The boundaries are **inclusive**. Changing `>=` to `>` and `<=` to `<` gives the
open band; both are valid, but the test must choose one. The test below uses the
closed band and checks the boundary values explicitly.

In [ ]:
@triton.jit
def m8_dead_zone_kernel(x_ptr, out_ptr, n_elements, lo, hi, BLOCK: tl.constexpr):
    pid = tl.program_id(0)
    offs = pid * BLOCK + tl.arange(0, BLOCK)
    mask = offs < n_elements
    x = tl.load(x_ptr + offs, mask=mask, other=0.0)

    keep = (x >= lo) & (x <= hi)
    y = tl.where(keep, x, 0.0)
    tl.store(out_ptr + offs, y, mask=mask)


def dead_zone(x, lo=-0.5, hi=0.5, BLOCK=256):
    out = torch.empty_like(x)
    m8_dead_zone_kernel[(triton.cdiv(x.numel(), BLOCK),)](
        x, out, x.numel(), lo, hi, BLOCK=BLOCK
    )
    return out


def test_dead_zone_kernel():
    require_gpu("m8 dead zone")
    x = torch.tensor([-2.0, -0.5, 0.0, 0.5, 2.0], device=DEVICE)
    ref = torch.where((x >= -0.5) & (x <= 0.5), x, torch.zeros_like(x))
    check("dead zone boundaries", dead_zone(x), ref)

    torch.manual_seed(0)
    x = torch.randn(1000, device=DEVICE)
    check("dead zone random", dead_zone(x),
          torch.where((x >= -0.5) & (x <= 0.5), x, torch.zeros_like(x)))


run_if_gpu(test_dead_zone_kernel)

# Part IV — Method 03: fused mask-select kernels

## 10. Soft-threshold: comparison, absolute value, and select in one pass

Soft-thresholding (the shrinkage operator) is

$$
S_t(x)=
\begin{cases}
x-t, & x\gt t,\\
0, & -t\le x\le t,\\
x+t, & x\lt -t.
\end{cases}
$$

It is the proximal operator of the L1 norm, the atom of sparse coding and
wavelet denoising, and a compact test of nested `where`. An equivalent form is

$$
S_t(x)=\operatorname{sign}(x)\,\max(\lvert x\rvert-t,0).
$$

The two forms must agree. The kernel below uses the nested-`where` form because
it exposes the three regions; the numerical cell checks it against the
absolute-value form.

**Fusion.** A naive implementation materializes three intermediates: the
comparison `x>t`, the comparison `x<-t`, and a temporary `|x|-t`. The fused
kernel keeps all of them in registers and writes only the final value. This is
the same argument as Chapter 7: the arithmetic is cheap, the extra round trips
are not.

In [ ]:
@triton.jit
def m8_soft_threshold_kernel(x_ptr, out_ptr, n_elements, t, BLOCK: tl.constexpr):
    pid = tl.program_id(0)
    offs = pid * BLOCK + tl.arange(0, BLOCK)
    mask = offs < n_elements
    x = tl.load(x_ptr + offs, mask=mask, other=0.0)

    y = tl.where(x > t, x - t,
                 tl.where(x < -t, x + t, 0.0))
    tl.store(out_ptr + offs, y, mask=mask)


def soft_threshold(x, t=0.5, BLOCK=256):
    out = torch.empty_like(x)
    m8_soft_threshold_kernel[(triton.cdiv(x.numel(), BLOCK),)](
        x, out, x.numel(), t, BLOCK=BLOCK
    )
    return out


def test_soft_threshold():
    require_gpu("m8 soft threshold")
    x = torch.tensor([-2.0, -0.5, 0.0, 0.5, 2.0], device=DEVICE)
    ref = torch.sign(x) * torch.clamp(x.abs() - 0.5, min=0.0)
    check("soft threshold spine", soft_threshold(x), ref)

    torch.manual_seed(0)
    x = torch.randn(1000, device=DEVICE) * 2.0
    ref = torch.sign(x) * torch.clamp(x.abs() - 0.5, min=0.0)
    check("soft threshold random", soft_threshold(x), ref)


run_if_gpu(test_soft_threshold)

## 11. A compile-time selector for the mask family

Production epilogues expose several mask modes behind one kernel. The mode is a
`tl.constexpr`, so each mode compiles to its own specialized body:

| `MODE` | Operation | Formula |
|---:|---|---|
| 0 | dead zone | $x\,[l\le x\le u]$ |
| 1 | soft threshold | $\operatorname{sign}(x)\max(\lvert x\rvert-t,0)$ |
| 2 | clamp | $\min(\max(x,l),u)$ |
| 3 | mask to zero | $x\,[l\le x\le u]$ |

Modes 0 and 3 are the same formula here; in a real epilogue mode 3 might write a
sentinel or skip the store. The point is the structure: one load, one scalar
expression, one store, no runtime switch.

**Shape rule.** The mode does not change the shape. Every arm returns one value
per input lane, and the store writes the same tile it loaded.

In [ ]:
MODE_DEAD_ZONE, MODE_SOFT, MODE_CLAMP, MODE_MASK_ZERO = range(4)


@triton.jit
def m8_mask_select_kernel(x_ptr, out_ptr, n_elements, lo, hi,
                          MODE: tl.constexpr, BLOCK: tl.constexpr):
    pid = tl.program_id(0)
    offs = pid * BLOCK + tl.arange(0, BLOCK)
    mask = offs < n_elements
    x = tl.load(x_ptr + offs, mask=mask, other=0.0)

    if MODE == MODE_DEAD_ZONE:
        y = tl.where((x >= lo) & (x <= hi), x, 0.0)
    elif MODE == MODE_SOFT:
        y = tl.where(x > hi, x - hi,
                     tl.where(x < -hi, x + hi, 0.0))
    elif MODE == MODE_CLAMP:
        y = tl.minimum(tl.maximum(x, lo), hi)
    else:
        y = tl.where((x >= lo) & (x <= hi), x, 0.0)

    tl.store(out_ptr + offs, y, mask=mask)


def mask_select(x, lo=-1.0, hi=1.0, mode=MODE_DEAD_ZONE, BLOCK=256):
    out = torch.empty_like(x)
    m8_mask_select_kernel[(triton.cdiv(x.numel(), BLOCK),)](
        x, out, x.numel(), lo, hi, MODE=mode, BLOCK=BLOCK
    )
    return out


def test_mask_select():
    require_gpu("m8 mask select")
    torch.manual_seed(0)
    x = torch.randn(1000, device=DEVICE) * 2.0
    check("mode dead zone", mask_select(x, -0.5, 0.5, MODE_DEAD_ZONE),
          torch.where((x >= -0.5) & (x <= 0.5), x, torch.zeros_like(x)))
    check("mode soft threshold", mask_select(x, -0.5, 0.5, MODE_SOFT),
          torch.sign(x) * torch.clamp(x.abs() - 0.5, min=0.0))
    check("mode clamp", mask_select(x, -0.5, 0.5, MODE_CLAMP),
          torch.clamp(x, -0.5, 0.5))


run_if_gpu(test_mask_select)

## 12. Broadcasting on the host, not in the kernel

Triton kernels see raw pointers and explicit offsets. A kernel does **not**
receive a shape and does **not** infer broadcasting from strides. The host must
do one of three things:

1. pass tensors that already have the same shape (expand and, if needed,
   `contiguous()`);
2. pass a 2-D grid and index the broadcast operand with the appropriate
   modulo/division;
3. pass the stride and build the address explicitly.

Option 1 is the simplest and the one used in this chapter. `tensor.expand` is a
view with a zero stride; it does not copy. The kernel, however, needs a
contiguous buffer, so `.contiguous()` copies unless the expanded view is already
contiguous. The numerical cell prints the strides so the copy is visible.

**Rule of thumb.** Broadcast on the host for clarity, and fuse only when the
copy or the extra launch is the bottleneck. A bias add is usually fused by
indexing the bias with `offs % N` in the kernel, not by materializing a full
`(M,N)` bias.

In [ ]:
X = torch.arange(6.0).reshape(2, 3)
bias = torch.tensor([10.0, 20.0, 30.0])
assert X.shape == (2, 3) and bias.shape == (3,)
assert X.stride() == (3, 1) and bias.stride() == (1,)

expanded = bias.expand(2, 3)
assert expanded.shape == (2, 3) and expanded.stride() == (0, 1)
assert not expanded.is_contiguous()          # a zero-stride view, not a copy

materialized = expanded.contiguous()
assert materialized.shape == (2, 3) and materialized.stride() == (3, 1)
assert materialized.data_ptr() != bias.data_ptr()
assert torch.equal(X + expanded, X + materialized)
print("broadcast view: zero stride is a view; contiguous() materializes it")

## 13. Applied example — the six-comparison bit code

A compact way to test all six predicates at once is to pack them into one
integer. With bit $k$ assigned to predicate $k$,

$$
\text{code}_i
=
\sum_{k=0}^{5} 2^k\,[x_i\bowtie_k 0],
$$

the code is unique for each combination of predicates. The kernel computes the
six masks and combines them with shifts instead of six separate output buffers.

**Why this is a useful test.** A bug in one predicate changes one bit, so the
failure is visible as a single wrong code rather than a wrong Boolean row. The
test below compares against the PyTorch expression built from the same six
predicates.

**Bit assignments.** `eq=1`, `ne=2`, `gt=4`, `ge=8`, `lt=16`, `le=32`.

In [ ]:
@triton.jit
def m8_bit_code_kernel(x_ptr, out_ptr, n_elements, BLOCK: tl.constexpr):
    pid = tl.program_id(0)
    offs = pid * BLOCK + tl.arange(0, BLOCK)
    mask = offs < n_elements
    x = tl.load(x_ptr + offs, mask=mask, other=0.0)

    code = (
        (x == 0.0).to(tl.int32) * 1
        + (x != 0.0).to(tl.int32) * 2
        + (x > 0.0).to(tl.int32) * 4
        + (x >= 1.0).to(tl.int32) * 8
        + (x < 0.0).to(tl.int32) * 16
        + (x <= -1.0).to(tl.int32) * 32
    )
    tl.store(out_ptr + offs, code, mask=mask)


def bit_code(x, BLOCK=256):
    out = torch.empty(x.numel(), device=x.device, dtype=torch.int32)
    m8_bit_code_kernel[(triton.cdiv(x.numel(), BLOCK),)](
        x, out, x.numel(), BLOCK=BLOCK
    )
    return out.reshape(x.shape)


def test_bit_code():
    require_gpu("m8 bit code")
    x = torch.tensor([-2.0, -1.0, -0.5, 0.0, 0.5, 1.0, 2.0], device=DEVICE)
    ref = (
        (x == 0).int() + (x != 0).int() * 2 + (x > 0).int() * 4
        + (x >= 1).int() * 8 + (x < 0).int() * 16 + (x <= -1).int() * 32
    )
    check("bit code spine", bit_code(x), ref)
    assert bit_code(x).tolist() == [50, 50, 18, 1, 6, 14, 14]
    print("bit code: valid bits:", (bit_code(x) <= 63).all().item())


run_if_gpu(test_bit_code)

## 14. Applied example — a masked reduction that never sees the tail

Masked loads are not only about safety; they also change the algebra of a
reduction. If a reduction follows the load, the masked lanes must not contribute
their `other` value to the result. The neutral element is the operation's
identity:

| Reduction | Neutral `other` | Why |
|---|---|---|
| sum | `0.0` | adding zero changes nothing |
| product | `1.0` | multiplying by one changes nothing |
| max | `-inf` | it cannot win |
| min | `+inf` | it cannot win |

The kernel below computes a masked mean of the positive values in a tile. It is
a small example of the rule: the mask guards the load, and the reduction uses
the same mask to count the lanes that exist.

$$
\bar x = \frac{\sum_i x_i\,[x_i\gt 0]}{\sum_i [x_i\gt 0]}.
$$

If no positive value exists, the kernel writes `0.0` instead of `0/0 = nan`.
This is a select on the denominator, not a branch.

In [ ]:
@triton.jit
def m8_positive_mean_kernel(x_ptr, out_ptr, n_elements, BLOCK: tl.constexpr):
    offs = tl.arange(0, BLOCK)
    mask = offs < n_elements
    x = tl.load(x_ptr + offs, mask=mask, other=0.0)

    positive = (x > 0.0) & mask
    total = tl.sum(tl.where(positive, x, 0.0), axis=0)
    count = tl.sum(positive.to(tl.int32), axis=0)
    mean = tl.where(count > 0, total / tl.maximum(count, 1), 0.0)
    tl.store(out_ptr, mean)


def positive_mean(x):
    out = torch.empty(1, device=x.device, dtype=x.dtype)
    BLOCK = triton.next_power_of_2(x.numel())
    m8_positive_mean_kernel[(1,)](x, out, x.numel(), BLOCK=BLOCK)
    return out[0]


def test_positive_mean():
    require_gpu("m8 positive mean")
    x = torch.tensor([-2.0, 1.0, 2.0, -3.0], device=DEVICE)
    check("positive mean", positive_mean(x), torch.tensor(1.5, device=DEVICE))
    z = torch.tensor([-2.0, -1.0], device=DEVICE)
    check("positive mean of none", positive_mean(z), torch.tensor(0.0, device=DEVICE))


run_if_gpu(test_positive_mean)

# Capstone — masked normalization

The capstone combines the comparison, the select, and the arithmetic:

$$
y_i =
\begin{cases}
x_i/d_i, & d_i\ne0,\\
0, & d_i=0.
\end{cases}
$$

This is the safe-division pattern that appears in normalization layers, attention
denominators, and ratio features. The naive form

```python
y = x / d
y = tl.where(d != 0, y, 0.0)
```

discards the `inf`/`nan` in the result but still evaluates `x/d` on every lane. The
guarded form replaces the denominator before the division:

```python
safe = tl.where(d != 0, d, 1.0)
y = tl.where(d != 0, x / safe, 0.0)
```

The second form never creates `inf` or `nan`, so it is safe even if the select
is later optimized or the result is reduced. This is the important difference
between masking a value and guarding an operation.

## Worked example

For $x=(2,3,4)$ and $d=(1,0,2)$:

$$
y=(2/1,\;0,\;4/2)=(2,0,2).
$$

The middle lane is selected to zero; the division by the replaced denominator
is $3/1=3$, which is discarded. The next cell checks exactly this and prints the
intermediate `safe` tensor.

In [ ]:
@triton.jit
def m8_masked_normalize_kernel(x_ptr, d_ptr, out_ptr, n_elements,
                               BLOCK: tl.constexpr):
    pid = tl.program_id(0)
    offs = pid * BLOCK + tl.arange(0, BLOCK)
    mask = offs < n_elements

    x = tl.load(x_ptr + offs, mask=mask, other=0.0)
    d = tl.load(d_ptr + offs, mask=mask, other=0.0)

    nonzero = d != 0.0
    safe_d = tl.where(nonzero, d, 1.0)
    y = tl.where(nonzero, x / safe_d, 0.0)
    tl.store(out_ptr + offs, y, mask=mask)


def masked_normalize(x, d, BLOCK=256):
    x = torch.as_tensor(x).contiguous()
    d = torch.as_tensor(d).contiguous()
    out = torch.empty_like(x)
    m8_masked_normalize_kernel[(triton.cdiv(x.numel(), BLOCK),)](
        x, d, out, x.numel(), BLOCK=BLOCK
    )
    return out


def test_masked_normalize():
    require_gpu("m8 masked normalize")
    x = torch.tensor([2.0, 3.0, 4.0, 5.0], device=DEVICE)
    d = torch.tensor([1.0, 0.0, 2.0, -1.0], device=DEVICE)
    ref = torch.where(d != 0, x / torch.where(d != 0, d, torch.ones_like(d)),
                      torch.zeros_like(x))
    check("masked normalize", masked_normalize(x, d), ref)
    assert masked_normalize(x, d).tolist() == [2.0, 0.0, 2.0, -5.0]

    # The naive order still gives the right value here, but it creates nan first.
    with torch.no_grad():
        naive = torch.where(d != 0, x / d, torch.zeros_like(x))
    print("naive form result:", naive.tolist())
    print("non-finite before the select:", (~torch.isfinite(x / d)).tolist())


run_if_gpu(test_masked_normalize)

# Forward and backward

## 15. The derivative of a comparison and of `where`

A comparison is piecewise constant:

$$
m_i=[x_i\gt y_i]
\quad\Longrightarrow\quad
\frac{\partial m_i}{\partial x_i}=0,
\qquad
\frac{\partial m_i}{\partial y_i}=0
\quad\text{almost everywhere.}
$$

It has no classical derivative at the boundary $x_i=y_i$. Frameworks therefore
do not differentiate a comparison as a smooth function; they use it as a
condition and route the gradient through a select or a mask.

For a select,

$$
y_i=c_i\,a_i+(1-c_i)\,b_i,
$$

the local derivatives are

$$
\frac{\partial y_i}{\partial a_i}=c_i,
\qquad
\frac{\partial y_i}{\partial b_i}=1-c_i,
\qquad
\frac{\partial y_i}{\partial c_i}=a_i-b_i
\quad\text{(not used).}
$$

The last expression is the mathematical derivative with respect to a real
$c_i$, but $c_i$ is a Boolean produced by a comparison, so the chain stops
there: `c` has no gradient.

**Broadcasting and the backward.** If the forward broadcast a size-1 axis, the
backward **sums** the gradient over that axis. For a bias of shape `(N,)` added
to `(M,N)`, the bias gradient has shape `(N,)` and is a sum over the $M$ rows.
This is the Chapter 10 rule, and it applies to `where` as well: if `a` was
broadcast, `grad_a` is a reduction of `g*c`.

## 16. Worked backward — a piecewise-linear mask

Take

$$
y(x)=
\begin{cases}
x, & x\gt 0,\\
-2x, & x\le0.
\end{cases}
$$

This is `torch.where(x > 0, x, -2*x)`. Its derivative is $1$ on the positive
side and $-2$ on the non-positive side. At $x=0$ the function has a kink; the
framework chooses the branch selected by the condition (`x<=0` gives $-2$ at
zero in this expression). The next cell checks the forward values, the autograd
gradient, and the manual formula.

**Why this matters in a kernel.** The backward of a fused forward is another
elementwise kernel. It reads the upstream gradient $g$, recomputes the condition
(or loads the saved mask), and writes $g$ times the local derivative. The mask
is the only state that must be saved; the forward value is not needed for a
piecewise-linear activation.

In [ ]:
x = torch.tensor([-2.0, -1.0, 0.0, 1.0, 2.0], requires_grad=True)
y = torch.where(x > 0, x, -2.0 * x)
y.sum().backward()

assert y.detach().tolist() == [4.0, 2.0, -0.0, 1.0, 2.0]
assert x.grad.tolist() == [-2.0, -2.0, -2.0, 1.0, 1.0]
manual = torch.where(x.detach() > 0, torch.ones_like(x), -2.0 * torch.ones_like(x))
assert torch.equal(x.grad, manual)
print("backward: where routes the gradient to the chosen arm")

## 17. A backward kernel for the dead zone

The dead-zone forward is $y=x\,[l\le x\le u]$. Its derivative is the band mask:

$$
\frac{dy}{dx}=[l\le x\le u].
$$

If the forward is fused, the backward can recompute the mask from $x$ instead of
saving it. The kernel below reads $x$ and the upstream gradient $g$, recomputes
the band, and writes $g$ inside the band and zero outside. This is the same
traffic pattern as the forward: three vector reads/writes, one predicate, no
reduction.

**Boundary convention.** At $x=l$ or $x=u$ the mathematical derivative is not
unique. This kernel uses $1$ at both boundaries (the closed band). The test uses
the same convention; if the framework uses the open band, the two differ exactly
at the boundary.

In [ ]:
@triton.jit
def m8_dead_zone_backward_kernel(x_ptr, g_ptr, dx_ptr, n_elements, lo, hi,
                                 BLOCK: tl.constexpr):
    pid = tl.program_id(0)
    offs = pid * BLOCK + tl.arange(0, BLOCK)
    mask = offs < n_elements
    x = tl.load(x_ptr + offs, mask=mask, other=0.0)
    g = tl.load(g_ptr + offs, mask=mask, other=0.0)

    inside = (x >= lo) & (x <= hi)
    dx = tl.where(inside, g, 0.0)
    tl.store(dx_ptr + offs, dx, mask=mask)


def dead_zone_backward(x, g, lo=-0.5, hi=0.5, BLOCK=256):
    dx = torch.empty_like(x)
    m8_dead_zone_backward_kernel[(triton.cdiv(x.numel(), BLOCK),)](
        x, g, dx, x.numel(), lo, hi, BLOCK=BLOCK
    )
    return dx


def test_dead_zone_backward():
    require_gpu("m8 dead zone backward")
    x = torch.tensor([-2.0, -0.5, 0.0, 0.5, 2.0], device=DEVICE)
    g = torch.ones_like(x)
    ref = torch.where((x >= -0.5) & (x <= 0.5), g, torch.zeros_like(g))
    check("dead zone backward", dead_zone_backward(x, g), ref)


run_if_gpu(test_dead_zone_backward)

# Pitfalls, checks, and summary

## Pitfalls

| Pitfall | What happens | Fix |
|---|---|---|
| `x > 0 & x < 1` | `&` binds before the comparisons | `(x > 0) & (x < 1)` |
| `and` / `or` / `not` on tensors | raises, or short-circuits a scalar | use `&`, `\|`, `~` |
| `~` on an integer tensor | bitwise complement, not a Boolean mask | cast to `bool` first |
| comparing floats with `==` | exact equality fails after rounding | compare `abs(x-y) <= eps` |
| `nan` in a comparison | all ordered comparisons are false; `!=` is true | decide the NaN policy explicitly |
| storing a predicate without a cast | dtype mismatch or four-byte mask | `.to(tl.int8)` |
| forgetting the tail mask | out-of-bounds read/write | `mask = offs < n_elements` |
| `tl.where` with a dangerous arm | `inf`/`nan` computed before selection | guard the input, not the output |
| host/kernel broadcast mismatch | kernel reads the wrong stride or length | expand + `contiguous()`, or index explicitly |
| `a & b` on integer masks | bitwise AND on 0/1 happens to work, but is not a predicate | keep masks Boolean until the cast |

## Shape-rule recap

| Situation | Rule | Result |
|---|---|---|
| `x` and `y` same shape | every dimension equal | same shape |
| `(4,3)` with `(3,)` | right-align; size-1 broadcasts | `(4,3)` |
| `(4,3)` with `(4,1)` | size-1 broadcasts on the column axis | `(4,3)` |
| `(4,3)` with scalar | scalar broadcasts to any shape | `(4,3)` |
| `(4,3)` with `(3,4)` | axes 2 vs 3, neither is 1 | error |
| `tl.where(c,a,b)` | broadcast all three | broadcast shape |
| kernel operands | same tile length unless the host expands | same length |

## Self-check

1. Write the six masks for $x=(-2,-1,0,1,2)$ and the 6-bit code for each value.
2. Why is `x > -1 & x < 1` wrong? Give the parse and the fix.
3. Prove De Morgan's law lane by lane for the band mask.
4. What are the shapes of `(4,3) + (3,)`, `(4,3) + (4,1)`, and `(4,3) + (3,4)`?
5. Why does `tl.where(c, x/y, 0)` still evaluate `x/y`? Rewrite it safely.
6. Derive the gradient of `where(x>0, x, -2*x)` at $x=1$ and $x=-1$.
7. What is the derivative of a comparison? Why is the question ill-posed at the boundary?
8. In the dead-zone kernel, which lanes does the comparison execute on? Which lanes does the store write?

## Summary

- A comparison is an elementwise map to a predicate tile; it does not branch.
- Masks compose with `&`, `|`, and `~`; parentheses are mandatory.
- `tl.where` selects lane by lane and evaluates both arms.
- The shape rule is the broadcast rule for comparisons and for `tl.where`.
- In a kernel, the host is responsible for same-length buffers or explicit
  indexing; Triton does not infer a broadcast from strides.
- A mask can guard memory (load/store) or select a value (`where`); they are
  different uses of the same predicate.
- The backward of `where` routes the gradient through the chosen arm; a
  comparison has zero derivative almost everywhere.

**Next:** [Chapter 9 — Activation Functions](../ch09_activations/09_complete.ipynb)

[Index](../../00_index.ipynb) · [Chapter README](README.md) · [← Previous](../ch07_arithmetic/07_complete.ipynb)

# Exercises

1. **Bit code.** Extend the bit-code kernel with a seventh bit for
   `abs(x) <= 1`. What is the maximum code now, and which values of the spine
   change?
2. **Open versus closed.** Rewrite the dead zone with strict inequalities and
   list the spine values whose output changes. Which convention does PyTorch's
   `torch.where` use? (It uses whichever predicate you write.)
3. **Operator trap.** Evaluate `x > -1 & x < 1` in Python for a tensor and
   explain the parse. Then write the parenthesized version.
4. **De Morgan.** Implement the complement of a band in two ways and check
   equality on a random tensor. Which form is cheaper in instructions?
5. **Broadcast.** Add a bias of shape `(3,)` to a matrix `(2,3)` in a Triton
   kernel by indexing with `offs % 3`. Compare the traffic with the host-side
   `expand().contiguous()` version.
6. **Safe division.** Write both orders (select after division versus guard
   before division) and feed a denominator of zero. Print the intermediate
   values; explain why the first creates `nan`.
7. **NaN policy.** Compare `x == x`, `x != x`, `x < 0`, and `x > 0` for
   `x = nan`. Which predicate identifies a NaN?
8. **Masked reduction.** Modify the positive-mean kernel to use `other=-inf`
   and a max instead of a sum. Which neutral element is correct, and why?
9. **Backward.** Derive the gradient of the dead zone at the boundaries and
   compare the closed-band convention with PyTorch's autograd on a tensor that
   hits the boundary.
10. **Fusion.** Fuse the dead zone and the soft threshold into one kernel with a
    `MODE` selector. Count the loads and stores of the fused version versus two
    separate launches for $N=2^{20}$ and `float32`.

In [ ]:
print("Running CPU tests ...")
test_worked_example()
test_naive_reference()

print("\nRunning GPU tests (skipped without CUDA/ROCm) ...")
for fn in (
    test_compare_kernel,
    test_where_kernel,
    test_dead_zone_kernel,
    test_soft_threshold,
    test_mask_select,
    test_bit_code,
    test_positive_mean,
    test_masked_normalize,
    test_dead_zone_backward,
):
    run_if_gpu(fn)